<a href="https://colab.research.google.com/github/mrdbourke/pytorch-deep-learning/blob/main/extras/exercises/05_pytorch_going_modular_exercise_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05. PyTorch Going Modular Exercises

Welcome to the 05. PyTorch Going Modular exercise template notebook.

There are several questions in this notebook and it's your goal to answer them by writing Python and PyTorch code.

> **Note:** There may be more than one solution to each of the exercises, don't worry too much about the *exact* right answer. Try to write some code that works first and then improve it if you can.

## Resources and solutions

* These exercises/solutions are based on [section 05. PyTorch Going Modular](https://www.learnpytorch.io/05_pytorch_going_modular/) of the Learn PyTorch for Deep Learning course by Zero to Mastery.

**Solutions:** 

Try to complete the code below *before* looking at these.

* See a live [walkthrough of the solutions (errors and all) on YouTube](https://youtu.be/ijgFhMK3pp4).
* See an example [solutions notebook for these exercises on GitHub](https://github.com/mrdbourke/pytorch-deep-learning/blob/main/extras/solutions/05_pytorch_going_modular_exercise_solutions.ipynb).

## 1. Turn the code to get the data (from section 1. Get Data) into a Python script, such as `get_data.py`.

* When you run the script using `python get_data.py` it should check if the data already exists and skip downloading if it does.
* If the data download is successful, you should be able to access the `pizza_steak_sushi` images from the `data` directory.

In [1]:
import os
os.makedirs("going_modular_exercise", exist_ok=True)

In [2]:
%%writefile going_modular_exercise/get_data.py
"""
Downloads the pizza, steak, sushi image dataset (10% of Food101) into data/pizza_steak_sushi.
Skips the download if the data already exists.

Usage: python get_data.py
"""
import os
import zipfile
from pathlib import Path

import requests

DATA_URL = "https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip"

# Setup path to data folder
data_path = Path("data/")
image_path = data_path / "pizza_steak_sushi"

# If the image folder doesn't exist, download it and prepare it...
if image_path.is_dir():
    print(f"[INFO] {image_path} directory exists, skipping download.")
else:
    print(f"[INFO] Did not find {image_path} directory, creating one...")
    image_path.mkdir(parents=True, exist_ok=True)

    # Download pizza, steak, sushi data
    zip_path = data_path / "pizza_steak_sushi.zip"
    with open(zip_path, "wb") as f:
        print(f"[INFO] Downloading pizza, steak, sushi data from {DATA_URL}...")
        request = requests.get(DATA_URL)
        request.raise_for_status()
        f.write(request.content)

    # Unzip pizza, steak, sushi data
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        print("[INFO] Unzipping pizza, steak, sushi data...")
        zip_ref.extractall(image_path)

    # Remove zip file
    os.remove(zip_path)

print(f"[INFO] Train images: {len(list((image_path / 'train').glob('*/*.jpg')))} | "
      f"Test images: {len(list((image_path / 'test').glob('*/*.jpg')))}")

Writing going_modular_exercise/get_data.py


In [3]:
# Example running of get_data.py
import sys
!"{sys.executable}" going_modular_exercise/get_data.py

[INFO] data\pizza_steak_sushi directory exists, skipping download.
[INFO] Train images: 225 | Test images: 75


## 2. Use [Python's `argparse` module](https://docs.python.org/3/library/argparse.html) to be able to send the `train.py` custom hyperparameter values for training procedures.
* Add an argument flag for using a different:
  * Training/testing directory
  * Learning rate
  * Batch size
  * Number of epochs to train for
  * Number of hidden units in the TinyVGG model
    * Keep the default values for each of the above arguments as what they already are (as in notebook 05).
* For example, you should be able to run something similar to the following line to train a TinyVGG model with a learning rate of 0.003 and a batch size of 64 for 20 epochs: `python train.py --learning_rate 0.003 batch_size 64 num_epochs 20`.
* **Note:** Since `train.py` leverages the other scripts we created in section 05, such as, `model_builder.py`, `utils.py` and `engine.py`, you'll have to make sure they're available to use too. You can find these in the [`going_modular` folder on the course GitHub](https://github.com/mrdbourke/pytorch-deep-learning/tree/main/going_modular/going_modular). 

In [4]:
%%writefile going_modular_exercise/data_setup.py
"""
Contains functionality for creating PyTorch DataLoaders for
image classification data.
"""
import os

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# On Windows every DataLoader worker is a new process that has to re-import torch,
# which is slower than loading these small images in the main process.
NUM_WORKERS = 0 if os.name == "nt" else os.cpu_count()

def create_dataloaders(
    train_dir: str,
    test_dir: str,
    transform: transforms.Compose,
    batch_size: int,
    num_workers: int=NUM_WORKERS
):
  """Creates training and testing DataLoaders.

  Takes in a training directory and testing directory path and turns
  them into PyTorch Datasets and then into PyTorch DataLoaders.

  Args:
    train_dir: Path to training directory.
    test_dir: Path to testing directory.
    transform: torchvision transforms to perform on training and testing data.
    batch_size: Number of samples per batch in each of the DataLoaders.
    num_workers: An integer for number of workers per DataLoader.

  Returns:
    A tuple of (train_dataloader, test_dataloader, class_names).
    Where class_names is a list of the target classes.
  """
  # Use ImageFolder to create dataset(s)
  train_data = datasets.ImageFolder(train_dir, transform=transform)
  test_data = datasets.ImageFolder(test_dir, transform=transform)

  # Get class names
  class_names = train_data.classes

  # Turn images into data loaders
  train_dataloader = DataLoader(
      train_data,
      batch_size=batch_size,
      shuffle=True,
      num_workers=num_workers,
  )
  test_dataloader = DataLoader(
      test_data,
      batch_size=batch_size,
      shuffle=False,
      num_workers=num_workers,
  )

  return train_dataloader, test_dataloader, class_names

Writing going_modular_exercise/data_setup.py


In [5]:
%%writefile going_modular_exercise/model_builder.py
"""
Contains PyTorch model code to instantiate a TinyVGG model.
"""
import torch
from torch import nn

class TinyVGG(nn.Module):
  """Creates the TinyVGG architecture.

  Replicates the TinyVGG architecture from the CNN explainer website in PyTorch.
  See the original architecture here: https://poloclub.github.io/cnn-explainer/

  Args:
    input_shape: An integer indicating number of input channels.
    hidden_units: An integer indicating number of hidden units between layers.
    output_shape: An integer indicating number of output units.
  """
  def __init__(self, input_shape: int, hidden_units: int, output_shape: int) -> None:
      super().__init__()
      self.conv_block_1 = nn.Sequential(
          nn.Conv2d(in_channels=input_shape, out_channels=hidden_units, kernel_size=3, stride=1, padding=0),
          nn.ReLU(),
          nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=3, stride=1, padding=0),
          nn.ReLU(),
          nn.MaxPool2d(kernel_size=2, stride=2)
      )
      self.conv_block_2 = nn.Sequential(
          nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=0),
          nn.ReLU(),
          nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=0),
          nn.ReLU(),
          nn.MaxPool2d(2)
      )
      self.classifier = nn.Sequential(
          nn.Flatten(),
          # 64x64 input images -> 13x13 feature maps after the two conv blocks
          nn.Linear(in_features=hidden_units*13*13, out_features=output_shape)
      )

  def forward(self, x: torch.Tensor):
      return self.classifier(self.conv_block_2(self.conv_block_1(x)))

Writing going_modular_exercise/model_builder.py


In [6]:
%%writefile going_modular_exercise/engine.py
"""
Contains functions for training and testing a PyTorch model.
"""
import torch

from tqdm.auto import tqdm
from typing import Dict, List, Tuple

def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device: torch.device) -> Tuple[float, float]:
  """Trains a PyTorch model for a single epoch, returns (train_loss, train_accuracy)."""
  model.train()
  train_loss, train_acc = 0, 0
  for batch, (X, y) in enumerate(dataloader):
      X, y = X.to(device), y.to(device)
      y_pred = model(X)
      loss = loss_fn(y_pred, y)
      train_loss += loss.item()
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()
      y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
      train_acc += (y_pred_class == y).sum().item()/len(y_pred)
  return train_loss / len(dataloader), train_acc / len(dataloader)

def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              device: torch.device) -> Tuple[float, float]:
  """Tests a PyTorch model for a single epoch, returns (test_loss, test_accuracy)."""
  model.eval()
  test_loss, test_acc = 0, 0
  with torch.inference_mode():
      for batch, (X, y) in enumerate(dataloader):
          X, y = X.to(device), y.to(device)
          test_pred_logits = model(X)
          loss = loss_fn(test_pred_logits, y)
          test_loss += loss.item()
          test_pred_labels = test_pred_logits.argmax(dim=1)
          test_acc += ((test_pred_labels == y).sum().item()/len(test_pred_labels))
  return test_loss / len(dataloader), test_acc / len(dataloader)

def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          epochs: int,
          device: torch.device) -> Dict[str, List]:
  """Trains and tests a PyTorch model, returns a dictionary of per-epoch metrics."""
  results = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
  model.to(device)
  for epoch in tqdm(range(epochs)):
      train_loss, train_acc = train_step(model=model,
                                         dataloader=train_dataloader,
                                         loss_fn=loss_fn,
                                         optimizer=optimizer,
                                         device=device)
      test_loss, test_acc = test_step(model=model,
                                      dataloader=test_dataloader,
                                      loss_fn=loss_fn,
                                      device=device)
      print(
        f"Epoch: {epoch+1} | "
        f"train_loss: {train_loss:.4f} | "
        f"train_acc: {train_acc:.4f} | "
        f"test_loss: {test_loss:.4f} | "
        f"test_acc: {test_acc:.4f}"
      )
      results["train_loss"].append(train_loss)
      results["train_acc"].append(train_acc)
      results["test_loss"].append(test_loss)
      results["test_acc"].append(test_acc)
  return results

Writing going_modular_exercise/engine.py


In [7]:
%%writefile going_modular_exercise/utils.py
"""
Contains various utility functions for PyTorch model training and saving.
"""
import torch
from pathlib import Path

def save_model(model: torch.nn.Module,
               target_dir: str,
               model_name: str):
  """Saves a PyTorch model's state_dict() to target_dir/model_name (.pth or .pt)."""
  target_dir_path = Path(target_dir)
  target_dir_path.mkdir(parents=True, exist_ok=True)

  assert model_name.endswith(".pth") or model_name.endswith(".pt"), "model_name should end with '.pt' or '.pth'"
  model_save_path = target_dir_path / model_name

  print(f"[INFO] Saving model to: {model_save_path}")
  torch.save(obj=model.state_dict(), f=model_save_path)

Writing going_modular_exercise/utils.py


In [8]:
%%writefile going_modular_exercise/train.py
"""
Trains a PyTorch image classification model using device-agnostic code.

Example usage:
  python train.py --learning_rate 0.003 --batch_size 64 --num_epochs 20
"""
import argparse

import torch
from torchvision import transforms

import data_setup, engine, model_builder, utils

def get_args():
  parser = argparse.ArgumentParser(description="Train a TinyVGG model on image folders.")
  parser.add_argument("--train_dir", type=str, default="data/pizza_steak_sushi/train",
                      help="directory with training images (one sub-folder per class)")
  parser.add_argument("--test_dir", type=str, default="data/pizza_steak_sushi/test",
                      help="directory with testing images (one sub-folder per class)")
  parser.add_argument("--learning_rate", type=float, default=0.001,
                      help="learning rate for the Adam optimizer")
  parser.add_argument("--batch_size", type=int, default=32,
                      help="number of samples per batch")
  parser.add_argument("--num_epochs", type=int, default=5,
                      help="number of epochs to train for")
  parser.add_argument("--hidden_units", type=int, default=10,
                      help="number of hidden units in the TinyVGG layers")
  parser.add_argument("--model_name", type=str, default="05_going_modular_script_mode_tinyvgg_model.pth",
                      help="filename to save the trained model under models/")
  return parser.parse_args()

def main():
  args = get_args()
  print(f"[INFO] Training a model for {args.num_epochs} epochs with batch size {args.batch_size}, "
        f"{args.hidden_units} hidden units and a learning rate of {args.learning_rate}")
  print(f"[INFO] Training data file: {args.train_dir}")
  print(f"[INFO] Testing data file: {args.test_dir}")

  # Setup target device
  device = "cuda" if torch.cuda.is_available() else "cpu"

  # Create transforms
  data_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
  ])

  # Create DataLoaders with help from data_setup.py
  train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
      train_dir=args.train_dir,
      test_dir=args.test_dir,
      transform=data_transform,
      batch_size=args.batch_size
  )

  # Create model with help from model_builder.py
  torch.manual_seed(42)
  model = model_builder.TinyVGG(
      input_shape=3,
      hidden_units=args.hidden_units,
      output_shape=len(class_names)
  ).to(device)

  # Set loss and optimizer
  loss_fn = torch.nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(model.parameters(),
                               lr=args.learning_rate)

  # Start training with help from engine.py
  engine.train(model=model,
               train_dataloader=train_dataloader,
               test_dataloader=test_dataloader,
               loss_fn=loss_fn,
               optimizer=optimizer,
               epochs=args.num_epochs,
               device=device)

  # Save the model with help from utils.py
  utils.save_model(model=model,
                   target_dir="models",
                   model_name=args.model_name)

# The guard is required on Windows/macOS: DataLoader workers re-import this file
if __name__ == "__main__":
  main()

Writing going_modular_exercise/train.py


In [9]:
# Example running of train.py
!"{sys.executable}" going_modular_exercise/train.py --num_epochs 5 --batch_size 128 --hidden_units 128 --learning_rate 0.0003

[INFO] Training a model for 5 epochs with batch size 128, 128 hidden units and a learning rate of 0.0003
[INFO] Training data file: data/pizza_steak_sushi/train
[INFO] Testing data file: data/pizza_steak_sushi/test
Epoch: 1 | train_loss: 1.1031 | train_acc: 0.3137 | test_loss: 1.0937 | test_acc: 0.3333
Epoch: 2 | train_loss: 1.0941 | train_acc: 0.3434 | test_loss: 1.0972 | test_acc: 0.3467
Epoch: 3 | train_loss: 1.0792 | train_acc: 0.4221 | test_loss: 1.0756 | test_acc: 0.4000
Epoch: 4 | train_loss: 1.0479 | train_acc: 0.5049 | test_loss: 1.0435 | test_acc: 0.4667
Epoch: 5 | train_loss: 0.9918 | train_acc: 0.5310 | test_loss: 1.0377 | test_acc: 0.4533
[INFO] Saving model to: models\05_going_modular_script_mode_tinyvgg_model.pth



100%|##########| 5/5 [00:38<00:00,  7.77s/it]


In [10]:
# Check the --help message generated by argparse
!"{sys.executable}" going_modular_exercise/train.py --help

usage: train.py [-h] [--train_dir TRAIN_DIR] [--test_dir TEST_DIR]
                [--learning_rate LEARNING_RATE] [--batch_size BATCH_SIZE]
                [--num_epochs NUM_EPOCHS] [--hidden_units HIDDEN_UNITS]
                [--model_name MODEL_NAME]

Train a TinyVGG model on image folders.

options:
  -h, --help            show this help message and exit
  --train_dir TRAIN_DIR
                        directory with training images (one sub-folder per
                        class)
  --test_dir TEST_DIR   directory with testing images (one sub-folder per
                        class)
  --learning_rate LEARNING_RATE
                        learning rate for the Adam optimizer
  --batch_size BATCH_SIZE
                        number of samples per batch
  --num_epochs NUM_EPOCHS
                        number of epochs to train for
  --hidden_units HIDDEN_UNITS
                        number of hidden units in the TinyVGG layers
  --model_name MODEL_NAME
                        fil

## 3. Create a Python script to predict (such as `predict.py`) on a target image given a file path with a saved model.

* For example, you should be able to run the command `python predict.py some_image.jpeg` and have a trained PyTorch model predict on the image and return its prediction.
* To see example prediction code, check out the [predicting on a custom image section in notebook 04](https://www.learnpytorch.io/04_pytorch_custom_datasets/#113-putting-custom-image-prediction-together-building-a-function). 
* You may also have to write code to load in a trained model.

In [11]:
%%writefile going_modular_exercise/predict.py
"""
Predicts the class of a target image with a trained TinyVGG model.

Example usage:
  python predict.py --image data/pizza_steak_sushi/test/sushi/175783.jpg
"""
import argparse

import torch
import torchvision
from torchvision import transforms

import model_builder

def get_args():
  parser = argparse.ArgumentParser(description="Predict on a target image with a trained TinyVGG model.")
  parser.add_argument("--image", type=str, required=True,
                      help="filepath of the target image to predict on")
  parser.add_argument("--model_path", type=str, default="models/05_going_modular_script_mode_tinyvgg_model.pth",
                      help="filepath of the trained model state_dict to load")
  parser.add_argument("--class_names", type=str, nargs="+", default=["pizza", "steak", "sushi"],
                      help="class names in the same order as the model was trained on")
  return parser.parse_args()

def load_model(model_path: str, device: torch.device) -> torch.nn.Module:
  """Loads a TinyVGG state_dict, reading the model sizes from the saved weights."""
  state_dict = torch.load(model_path, map_location=device)
  # First conv weight has shape [hidden_units, input_channels, 3, 3]
  hidden_units = state_dict["conv_block_1.0.weight"].shape[0]
  # Last linear weight has shape [output_shape, hidden_units*13*13]
  output_shape = state_dict["classifier.1.weight"].shape[0]
  model = model_builder.TinyVGG(input_shape=3,
                                hidden_units=hidden_units,
                                output_shape=output_shape).to(device)
  model.load_state_dict(state_dict)
  print(f"[INFO] Loaded model from {model_path} ({hidden_units} hidden units)")
  return model

def main():
  args = get_args()
  device = "cuda" if torch.cuda.is_available() else "cpu"

  model = load_model(args.model_path, device)

  # Load the image, turn it into float values between 0 and 1 and resize it like the training data
  image = torchvision.io.read_image(args.image).type(torch.float32) / 255.
  transform = transforms.Resize(size=(64, 64))
  image = transform(image)

  # Predict on the image (add a batch dimension first)
  model.eval()
  with torch.inference_mode():
    pred_logits = model(image.unsqueeze(dim=0).to(device))
  pred_prob = torch.softmax(pred_logits, dim=1)
  pred_label = torch.argmax(pred_prob, dim=1).item()

  print(f"[INFO] Predicting on {args.image}")
  print(f"[INFO] Pred class: {args.class_names[pred_label]}, Pred prob: {pred_prob.max():.3f}")

if __name__ == "__main__":
  main()

Writing going_modular_exercise/predict.py


In [12]:
# Example running of predict.py
!"{sys.executable}" going_modular_exercise/predict.py --image data/pizza_steak_sushi/test/sushi/175783.jpg

[INFO] Loaded model from models/05_going_modular_script_mode_tinyvgg_model.pth (128 hidden units)
[INFO] Predicting on data/pizza_steak_sushi/test/sushi/175783.jpg
[INFO] Pred class: sushi, Pred prob: 0.405


In [13]:
# Try it on a few more images from the test set
!"{sys.executable}" going_modular_exercise/predict.py --image data/pizza_steak_sushi/test/pizza/2124579.jpg
!"{sys.executable}" going_modular_exercise/predict.py --image data/pizza_steak_sushi/test/steak/100274.jpg

[INFO] Loaded model from models/05_going_modular_script_mode_tinyvgg_model.pth (128 hidden units)
[INFO] Predicting on data/pizza_steak_sushi/test/pizza/2124579.jpg
[INFO] Pred class: steak, Pred prob: 0.494


[INFO] Loaded model from models/05_going_modular_script_mode_tinyvgg_model.pth (128 hidden units)
[INFO] Predicting on data/pizza_steak_sushi/test/steak/100274.jpg
[INFO] Pred class: steak, Pred prob: 0.501
